# AdventureWorks predictions with Kumo Relational

Kumo Relational is a foundation model for relational data.
It answers a predictive query over linked tables directly, with no feature engineering and no training.
This notebook asks it about the AdventureWorks sales tables in Unity Catalog: product demand, customer churn, product recommendations and a product attribute.
Forecasts made at a past date are checked against what actually sold, and the same forecasts are then asked through a metric view.

Requirements:

- The four AdventureWorks tables in one schema: `adventureworks_customers`, `adventureworks_products`, `adventureworks_sales_order_headers` and `adventureworks_sales_order_details`.
- A Model Serving endpoint that serves Kumo Relational.
- Serverless compute with environment 4 or later, or Databricks Runtime 17.3 LTS or later.
- Optional: a metric view over these tables.

Set `catalog`, `schema` and `endpoint` in the parameters at the top, and `metric_view` for the metric view section, then run all cells.

In [ ]:
dbutils.widgets.text('catalog', 'main', 'Catalog')
dbutils.widgets.text('schema', 'adventureworks', 'Schema')
dbutils.widgets.text('endpoint', '', 'Serving endpoint')
dbutils.widgets.text('metric_view', '', 'Metric view (optional)')
dbutils.widgets.text('warehouse_id', '', 'SQL warehouse ID (optional)')
dbutils.widgets.text('output_table', '', 'Output table (optional)')
dbutils.widgets.text(
    'install',
    'structured-data-models-client[relational,databricks,databricks-serving]',
    'pip install',
)
dbutils.widgets.dropdown('ci_output', 'false', ['false', 'true'], 'CI output')

import numpy

install = dbutils.widgets.get('install')
keep_numpy = f'numpy=={numpy.__version__}'  # keep the runtime's NumPy

In [ ]:
%pip install --only-binary=:all: $install $keep_numpy

In [ ]:
dbutils.library.restartPython()

## Connect

The client reads the tables in place, as you, and sends only the rows each prediction needs to the serving endpoint.

In [ ]:
from sdm_client import (
    KumoRelational,
    PredictiveQuery,
    Relationship,
    Schema,
    SDMClient,
)
from sdm_connectors import DatabricksSource
from sdm_connectors.databricks import warehouse

catalog = dbutils.widgets.get('catalog')
schema_name = dbutils.widgets.get('schema')
endpoint = dbutils.widgets.get('endpoint')
metric_view = dbutils.widgets.get('metric_view')
warehouse_id = dbutils.widgets.get('warehouse_id')
output_table = dbutils.widgets.get('output_table')

connection = warehouse(warehouse_id) if warehouse_id else spark
source = DatabricksSource(connection, catalog=catalog, schema=schema_name)
client = SDMClient.for_databricks_serving(endpoint)
model = KumoRelational()

## Describe the tables

The model needs each table's primary key, how the tables link, and which tables record events over time.
These tables declare no keys in the catalog, so they are stated here.
`products` and `customers` get no time column, because they describe things rather than events.
Two columns that would mislead the model are left out: a random ID per row, and the time a row was last edited.

In [ ]:
schema = Schema.from_source(
    source,
    {
        'customers': 'adventureworks_customers',
        'products': 'adventureworks_products',
        'sales_order_headers': 'adventureworks_sales_order_headers',
        'sales_order_details': 'adventureworks_sales_order_details',
    },
    primary_keys={
        'customers': 'CustomerID',
        'products': 'ProductID',
        'sales_order_details': 'SalesOrderDetailID',
    },
    time_columns={
        'sales_order_headers': 'OrderDate',
        'sales_order_details': 'OrderDate',
        'products': None,
        'customers': None,
    },
    relationships=[
        Relationship(
            'sales_order_headers', 'CustomerID', 'customers', 'CustomerID'
        ),
        Relationship(
            'sales_order_details', 'CustomerID', 'customers', 'CustomerID'
        ),
        Relationship(
            'sales_order_details', 'ProductID', 'products', 'ProductID'
        ),
    ],
)
# rowguid is a random ID per row, so it tells the model nothing.
for table in schema.tables.values():
    del table['rowguid']
# ModifiedDate is when a row was last edited, which is later than the past date
# used below, and in sales_order_details it repeats OrderDate.
for name in ('products', 'customers', 'sales_order_details'):
    del schema[name]['ModifiedDate']

print(*schema.tables.values(), sep='\n')
print(schema.relationships)

## Choose what to score

The examples score the best-selling products and the most frequent customers, picked with plain SQL.
The helpers here also read product names and actual sales so the results are easy to read; none of this is needed to make a prediction.

In [ ]:
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd
from sdm_connectors import quote_ident

shown_warnings = set()


def show_once(message, category, filename, lineno, file=None, line=None):
    # Show each note about the data once, rather than with every prediction.
    if (str(message), category) not in shown_warnings:
        shown_warnings.add((str(message), category))
        text = warnings.formatwarning(message, category, filename, lineno, line)
        sys.stderr.write(text)


warnings.showwarning = show_once


def quoted(name):
    # A table, schema.table or catalog.schema.table name, completed from the
    # catalog and schema parameters and quoted for SQL.
    parts = name.split('.')
    parts = [catalog, schema_name][: 3 - len(parts)] + parts
    return '.'.join(quote_ident(part, char='`') for part in parts)


def sql(query):
    return spark.sql(query).toPandas()


def top(table, key, order, limit, until=None):
    # The `limit` IDs that rank highest by `order`, over the orders up to `until`.
    where = f"WHERE OrderDate <= '{until}'" if until is not None else ''
    ids = sql(
        f'SELECT {key} FROM {table} {where} GROUP BY {key} '
        f'ORDER BY {order} DESC, {key} LIMIT {limit}'
    )
    return ids[key].tolist()


def actual_sales(start, windows=1, column='OrderQty'):
    # Each product's total `column` in consecutive 30-day windows after `start`.
    frames = []
    for step in range(1, windows + 1):
        begin = start + pd.Timedelta(days=30 * (step - 1))
        end = begin + pd.Timedelta(days=30)
        frame = sql(
            f'SELECT ProductID AS id, sum({column}) AS actual FROM {details} '
            f"WHERE OrderDate > '{begin}' AND OrderDate <= '{end}' "
            'GROUP BY ProductID'
        )
        frames.append(frame.assign(forecast_step=step))
    return pd.concat(frames)


def compare(frame, actual=None):
    # Add each product's name, and its actual value in the same window.
    frame = frame.assign(product=frame['id'].map(products['Name']))
    if actual is not None:
        keys = ['id', 'forecast_step'] if 'forecast_step' in frame else ['id']
        frame = frame.merge(actual[[*keys, 'actual']], on=keys, how='left')
        frame['actual'] = frame['actual'].fillna(0)
    return frame


def ranked(frame):
    # One row per recommended product, in rank order.
    return pd.DataFrame(
        {
            'customer': row.id,
            'rank': rank,
            'product': products['Name'][item['candidate']],
            'score': item['score'],
        }
        for row in frame.itertuples()
        for rank, item in enumerate(row.rankings, start=1)
    )


details = quoted('adventureworks_sales_order_details')
headers = quoted('adventureworks_sales_order_headers')
latest = pd.Timestamp(
    spark.sql(f'SELECT max(OrderDate) FROM {details}').first()[0]
)
anchor = latest - pd.Timedelta(days=180)

top_products = top(details, 'ProductID', 'sum(OrderQty)', 5)
frequent_customers = top(headers, 'CustomerID', 'count(*)', 25)
# For the questions asked at the past date, picked from the orders up to it.
past_top_products = top(details, 'ProductID', 'sum(OrderQty)', 5, anchor)
past_popular_products = top(
    details, 'ProductID', 'count(DISTINCT CustomerID)', 10, anchor
)
past_frequent_customers = top(headers, 'CustomerID', 'count(*)', 3, anchor)

labelled = {*top_products, *past_top_products, *past_popular_products}
products = sql(
    f'SELECT ProductID, Name, Color FROM {quoted("adventureworks_products")} '
    f'WHERE ProductID IN ({", ".join(map(str, labelled))})'
).set_index('ProductID')
accounts = sql(
    f'SELECT CustomerID, AccountNumber '
    f'FROM {quoted("adventureworks_customers")} '
    f'WHERE CustomerID IN ({", ".join(map(str, frequent_customers))})'
).set_index('CustomerID')['AccountNumber']

print(f'Last order: {latest.date()}. Past date: {anchor.date()}.')

## Demand forecast

How many units of each best-selling product will sell in the next 30 days?

```
PREDICT SUM(sales_order_details.OrderQty, 0, 30, days)   ← units sold in the next 30 days
FOR EACH products.ProductID                              ← one prediction per product
```

The forecast starts at the last order in the data, and `last_30_days` shows what each product sold in the 30 days before it.

In [ ]:
DEMAND = (
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
    'FOR EACH products.ProductID'
)

request = PredictiveQuery(DEMAND, schema).materialize(indices=top_products)
demand = client.predict(request, model=model).to_pandas()

recent = actual_sales(latest - pd.Timedelta(days=30))
demand = compare(demand, recent).rename(columns={'actual': 'last_30_days'})
display(demand[['id', 'product', 'prediction', 'last_30_days']])

## Demand forecast at a past date

With `anchor_time`, the model sees only the data before that date, so its forecast can be checked against what actually sold.
Here the past date is 180 days before the last order.
The forecasts checked against actual sales pass `num_context_samples=2000`, so the model learns from more examples, taken at more past dates.

In [ ]:
request = PredictiveQuery(DEMAND, schema).materialize(
    indices=past_top_products, anchor_time=anchor, num_context_samples=2000
)
past_demand = client.predict(request, model=model).to_pandas()

past_demand = compare(past_demand, actual_sales(anchor))
display(past_demand[['id', 'product', 'prediction', 'actual']])
past_demand.plot.bar(
    x='product',
    y=['prediction', 'actual'],
    rot=20,
    title=f'Units sold in the 30 days after {anchor.date()}',
)
plt.show()

## Multi-period forecast

`FORECAST 3 TIMEFRAMES` predicts three consecutive 30-day windows from the past date.

```
PREDICT SUM(sales_order_details.OrderQty, 0, 30, days)   ← units sold in one 30-day window
FORECAST 3 TIMEFRAMES                                    ← three windows in a row
FOR EACH products.ProductID                              ← one prediction per product and window
```

`forecast_step` numbers the windows from 1, so step 1 asks the same question as the forecast above.

In [ ]:
request = PredictiveQuery(
    'PREDICT SUM(sales_order_details.OrderQty, 0, 30, days) '
    'FORECAST 3 TIMEFRAMES FOR EACH products.ProductID',
    schema,
).materialize(
    indices=past_top_products, anchor_time=anchor, num_context_samples=2000
)
forecast = client.predict(request, model=model).to_pandas()

forecast = compare(forecast, actual_sales(anchor, windows=3))
display(
    forecast.sort_values(['product', 'forecast_step'])[
        ['id', 'product', 'forecast_step', 'prediction', 'actual']
    ]
)

## Customer churn

Which of the most frequent customers will place no order in the next 90 days?

```
PREDICT COUNT(sales_order_headers.*, 0, 90, days) = 0   ← no orders in the next 90 days
FOR EACH customers.CustomerID                           ← one prediction per customer
```

This is a yes-or-no question, so the result has a probability for each answer.
`probability_True` is the probability that the customer places no order.

In [ ]:
request = PredictiveQuery(
    'PREDICT COUNT(sales_order_headers.*, 0, 90, days) = 0 '
    'FOR EACH customers.CustomerID',
    schema,
).materialize(indices=frequent_customers)
churn = client.predict(request, model=model).to_pandas()

churn['AccountNumber'] = churn['id'].map(accounts)
churn = churn.sort_values('probability_True', ascending=False)
display(churn[['id', 'AccountNumber', 'probability_True', 'prediction']])

## Product recommendations

Which products will each customer buy in the next 30 days?

```
PREDICT LIST_DISTINCT(sales_order_details.ProductID, 0, 30, days)   ← products bought in the next 30 days
RANK TOP 5                                                          ← the five most likely
FOR EACH customers.CustomerID                                       ← one list per customer
```

A ranking chooses from candidates you pass, here the ten products bought by the most customers up to the past date.
It is asked at the past date, for the three customers with the most orders up to it.
A candidate the model cannot score is left out and listed in `unscored_candidates`.

In [ ]:
request = PredictiveQuery(
    'PREDICT LIST_DISTINCT(sales_order_details.ProductID, 0, 30, days) '
    'RANK TOP 5 FOR EACH customers.CustomerID',
    schema,
).materialize(
    indices=past_frequent_customers,
    candidates=past_popular_products,
    anchor_time=anchor,
)
result = client.predict(request, model=model)
recommendations = result.to_pandas()

print('Unscored candidates:', list(result.unscored_candidates) or 'none')
display(ranked(recommendations))

## Attribute prediction

A query with no time window predicts a property of each entity rather than a future event.

```
PREDICT products.Color        ← the product's color
FOR EACH products.ProductID   ← one prediction per product
```

The result has the predicted color and a probability for each color, shown next to the recorded one.
On your own data, the same query fills in missing values.

In [ ]:
request = PredictiveQuery(
    'PREDICT products.Color FOR EACH products.ProductID', schema
).materialize(indices=top_products)
colors = client.predict(request, model=model).to_pandas()

colors = compare(colors)
colors['recorded'] = colors['id'].map(products['Color'])
probabilities = [
    column for column in colors if column.startswith('probability_')
]
display(colors[['id', 'product', 'recorded', 'prediction', *probabilities]])

## Predictions through a metric view

A Unity Catalog metric view already defines a fact table, its joins, and the dimensions and measures over them.
`source.semantic_model(name)` reads the view, and `Schema.from_source` turns it into a schema, so you can ask questions without describing the tables yourself.
The conversion notes give the predictive query aggregation that matches each measure.
The view used here is defined at the end of the notebook.

In [ ]:
if metric_view:
    view_schema = Schema.from_source(source.semantic_model(metric_view))
    print(*view_schema.tables.values(), sep='\n')
    print(view_schema.relationships)
    print(*view_schema.conversion_notes, sep='\n')
else:
    print('No metric view set; skipping this section.')

The demand forecast at the past date, asked through the view, next to the forecast from the tables and the actual sales.
The view has no `sales_order_headers` and fewer columns than the schema above, so the two forecasts can differ.

In [ ]:
if metric_view:
    request = PredictiveQuery(
        'PREDICT SUM(adventureworks_sales_order_details.OrderQty, 0, 30, days) '
        'FOR EACH products.ProductID',
        view_schema,
    ).materialize(
        indices=past_top_products, anchor_time=anchor, num_context_samples=2000
    )
    view_demand = client.predict(request, model=model).to_pandas()

    view_demand = view_demand.merge(
        past_demand[['id', 'product', 'prediction', 'actual']],
        on='id',
        suffixes=('_view', '_tables'),
    )
    display(
        view_demand[
            ['id', 'product', 'prediction_view', 'prediction_tables', 'actual']
        ]
    )

The view's `revenue` measure is `SUM(source.LineTotal)`, so its 30-day forecast is the query below.
It is shown next to the actual revenue in the 30 days after the past date.

In [ ]:
if metric_view:
    request = PredictiveQuery(
        'PREDICT SUM(adventureworks_sales_order_details.LineTotal, 0, 30, days) '
        'FOR EACH products.ProductID',
        view_schema,
    ).materialize(
        indices=past_top_products, anchor_time=anchor, num_context_samples=2000
    )
    view_revenue = client.predict(request, model=model).to_pandas()

    revenue = actual_sales(anchor, column='LineTotal')
    view_revenue = compare(view_revenue, revenue)
    display(view_revenue[['id', 'product', 'prediction', 'actual']])

## Save predictions

Set `output_table` to append every prediction above to a Delta table.
Each row holds the question name and one predicted entity as JSON, since the questions return different columns.

In [ ]:
predictions = {
    'demand_30d': demand,
    'demand_30d_past': past_demand,
    'demand_3_windows': forecast,
    'churn_90d': churn,
    'recommend_top5': recommendations,
    'product_color': colors,
}
if metric_view:
    predictions['view_demand_30d_past'] = view_demand
    predictions['view_revenue_30d_past'] = view_revenue

if output_table:
    from pyspark.sql.functions import current_timestamp

    rows = [
        (question, line)
        for question, frame in predictions.items()
        for line in frame.to_json(orient='records', lines=True).splitlines()
    ]
    (
        spark.createDataFrame(rows, 'question string, prediction string')
        .withColumn('scored_at', current_timestamp())
        .write.mode('append')
        .saveAsTable(quoted(output_table))
    )
    print(f'Appended {len(rows)} rows to {quoted(output_table)}.')

## Use your own data

- Map a short name to each of your tables in `Schema.from_source`, and state the primary keys and relationships the catalog does not declare.
- Name the time column of each table that records events, pass `None` for tables that describe things, and leave out columns that would mislead the model, such as random row IDs and edit times.
- Write a predictive query for each question, and pass the IDs to score as `indices`.

## Troubleshooting

- The first prediction is slow or times out: an endpoint that has scaled to zero takes a few minutes to start, so run the cell again once it is up.
- A request is refused as too large: Model Serving accepts up to 16 MB per request, so score fewer IDs per call or pass a smaller `num_context_samples`.
- A `StructuredDataError`: the client checks each query against the schema before sending it, and the message says what to change.
- An older runtime: reading through Spark needs serverless environment 4 or Databricks Runtime 17.3 LTS, and from Databricks Runtime 14.3 LTS you can set `warehouse_id` to read through a SQL warehouse instead.

## Metric view used in this notebook

The metric view section was tested with this view; replace `main.adventureworks` with your catalog and schema.

```sql
CREATE VIEW main.adventureworks.adventureworks_sales_metrics
WITH METRICS LANGUAGE YAML AS $$
version: 0.1
source: main.adventureworks.adventureworks_sales_order_details
joins:
  - name: customers
    source: main.adventureworks.adventureworks_customers
    on: source.CustomerID = customers.CustomerID
  - name: products
    source: main.adventureworks.adventureworks_products
    on: source.ProductID = products.ProductID
dimensions:
  - name: SalesOrderDetailID
    expr: source.SalesOrderDetailID
  - name: SalesOrderID
    expr: source.SalesOrderID
  - name: OrderDate
    expr: source.OrderDate
  - name: OrderQty
    expr: source.OrderQty
  - name: UnitPrice
    expr: source.UnitPrice
  - name: LineTotal
    expr: source.LineTotal
  - name: Discounted
    expr: source.UnitPriceDiscount > 0
  - name: TerritoryID
    expr: customers.TerritoryID
  - name: StoreID
    expr: customers.StoreID
  - name: Color
    expr: products.Color
  - name: ListPrice
    expr: products.ListPrice
  - name: ProductLine
    expr: products.ProductLine
  - name: ProductSubcategoryID
    expr: products.ProductSubcategoryID
measures:
  - name: total_quantity
    expr: SUM(source.OrderQty)
  - name: revenue
    expr: SUM(source.LineTotal)
  - name: order_count
    expr: COUNT(DISTINCT source.SalesOrderID)
$$
```

In [ ]:
client.close()
if warehouse_id:
    connection.close()

# Automated runs set ci_output to true and end here with every prediction as JSON.
if dbutils.widgets.get('ci_output') == 'true':
    import json
    import os
    import platform
    import sys
    from importlib.metadata import version

    environment = {
        'runtime': os.environ.get('DATABRICKS_RUNTIME_VERSION'),
        'python': sys.version.split()[0],
        'machine': platform.machine(),
        'client': version('structured-data-models-client'),
        'engine': version('structured-data-models-relational-engine'),
    }
    results = {
        name: frame.to_dict('records') for name, frame in predictions.items()
    }
    notes = view_schema.conversion_notes if metric_view else []
    dbutils.notebook.exit(
        json.dumps(
            {
                'environment': environment,
                'results': results,
                'view_notes': [str(note) for note in notes],
            },
            default=str,
        )
    )